# TP4 · Fêtes et consommation au Sénégal
**Python avancé, L3 Intelligence Artificielle, ISM** · Hamze Abdoulkarim Aden · Enseignant : Elias BADJI

> **Données fictives.** Toutes les données sont simulées à des fins pédagogiques. Elles ne décrivent pas le comportement réel des Sénégalais ni d'aucune communauté religieuse, et aucune conclusion sur une communauté n'est tirée de ce travail.

**Les quatre questions de la direction**
1. **Mesurer** : quels produits s'envolent avant chaque fête, de combien, et dans quelles villes ?
2. **Prévoir** : peut-on prévoir la demande de 2025 avec un an d'avance ?
3. **Comprendre** : quels ménages s'endettent après une fête, et pourquoi ?
4. **Agir** : quelles quantités prévoir pour les fêtes de 2026 ?

**Sommaire** : 0. Configuration · 1. Audit et nettoyage · 2. Variables d'événement · 3. Effet des fêtes · Bonus promotions · 4. Prévision à un an · Bonus quantiles · 5. Endettement et équité · 6. Profils de dépense · 7. Plan 2026

## Partie 0 · Configuration

In [1]:
# === Imports ===
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

# === Interrupteur : False = recharger ce qui est déjà sauvegardé, True = tout recalculer ===
RECALCULER = False

# === Reproductibilité : la même graine partout ===
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# === Chemins : le notebook est dans notebooks/, la racine du projet est le dossier parent ===
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOSSIER_RAW = RACINE / "data" / "raw"
DOSSIER_PROCESSED = RACINE / "data" / "processed"
DOSSIER_MODELES = RACINE / "models"
DOSSIER_RAPPORTS = RACINE / "reports"
DOSSIER_FIGURES = DOSSIER_RAPPORTS / "figures"

# Ces dossiers sont ignorés par git : on les recrée s'ils n'existent pas (après un clone par exemple)
for dossier in [DOSSIER_PROCESSED, DOSSIER_MODELES, DOSSIER_FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

# === Affichage ===
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100


# === Fonctions utilitaires ===
def sauver_figure(fig, nom):
    """Enregistre une figure en PNG dans reports/figures/ (nom sans extension)."""
    chemin = DOSSIER_FIGURES / f"{nom}.png"
    fig.savefig(chemin, dpi=150, bbox_inches="tight")
    print("Figure enregistrée :", chemin.name)


def chemin_parquet(nom):
    """Chemin d'un fichier intermédiaire dans data/processed/."""
    return DOSSIER_PROCESSED / f"{nom}.parquet"


def chemin_modele(nom):
    """Chemin d'un modèle sauvegardé avec joblib dans models/."""
    return DOSSIER_MODELES / f"{nom}.joblib"


def peut_recharger(*chemins):
    """True si RECALCULER vaut False ET que tous les fichiers demandés existent déjà."""
    return (not RECALCULER) and all(Path(c).exists() for c in chemins)


print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("Racine du projet :", RACINE)
print("Fichiers bruts :", sorted(p.name for p in DOSSIER_RAW.glob("*.csv")))

pandas 3.0.6 | scikit-learn 1.9.1
Racine du projet : c:\Users\User\Desktop\L3S5\Python avance\TP4\fetes-consommation-senegal
Fichiers bruts : ['calendrier_evenements.csv', 'categories.csv', 'enquete_menages.csv', 'magasins.csv', 'ventes_journalieres.csv']
